# 05 - MultiPPO 概念手写

本节目标: 你能解释 multi-critic 只是在 value/reward/advantage 维度多了 `K` 个分支。

In [ ]:
import torch

T = 4
N = 3
K = 4
gamma = 0.99
lam = 0.95

torch.manual_seed(2)
rewards = torch.randn(T, N, K)
values = torch.randn(T, N, K)
last_values = torch.randn(N, K)
dones = torch.zeros(T, N, 1)
weights = torch.tensor([2.5, 0.1, 1.0, 2.0])

print('rewards:', rewards.shape)
print('values:', values.shape)
print('weights:', weights.shape)

## 每个 critic 单独算 GAE

`K` 个 critic 就有 `K` 路 reward、value、return、advantage。

In [ ]:
advantages = torch.zeros(T, N, K)
returns = torch.zeros(T, N, K)
advantage = torch.zeros(N, K)

for step in reversed(range(T)):
    next_values = last_values if step == T - 1 else values[step + 1]
    not_done = 1.0 - dones[step].float()
    delta = rewards[step] + not_done * gamma * next_values - values[step]
    advantage = delta + not_done * gamma * lam * advantage
    advantages[step] = advantage
    returns[step] = advantage + values[step]

print('advantages:', advantages.shape)
print('returns:', returns.shape)

## actor 用 weighted advantage

critic 学各自 reward group。actor 更新时把多路 advantage 加权合成一路。

In [ ]:
weighted_advantages = torch.sum(advantages * weights.view(1, 1, K), dim=-1)
print('weighted_advantages:', weighted_advantages.shape)
print(weighted_advantages[0])

## 作业

1. 把 `K=5`, 自己设置 5 个 weights。
2. 解释 shared multi-head critic 和 5 个独立 critic 的区别。
3. 说明为什么 `weighted_advantages` shape 是 `[T, N]`, 不是 `[T, N, K]`。